# Session 1 — Register to Unity Catalog & Deploy Serving Endpoint

**Goal:** Take the best model from our MLflow experiment and:
1. Register it in the Unity Catalog Model Registry
2. Assign it the `champion` alias
3. Deploy it as a real-time serving endpoint
4. Query it with a sample customer record

In [0]:
%run ../utils/config

In [0]:
%pip install ../bundle/wheels/ databricks-feature-engineering --quiet

## Provide your Best Run ID
1. If you haven't already done so, 
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">paste the value of your **best_run_id** in the notebook parameter at the top of the page.</strong>
  <div style="color:#333;">


In [0]:
# Get the best_run_id from the notebook widget
dbutils.widgets.text("best_run_id", "", "Best Run ID (from 04_mlflow_experiment)")
best_run_id = dbutils.widgets.get("best_run_id")

if not best_run_id:
    raise ValueError("Please enter the best_run_id from 04_mlflow_experiment.ipynb")

model_name = f"{catalog}.{schema}.churn_classifier"
print(f"Model name  : {model_name}")
print(f"Best run ID : {best_run_id}")

## MLflow model registry
MLflow can utilize the registry of your choice.  By default in Databricks, MLflow will register models in the workspace files space.  In the following cell, we instruct MLflow to register the model with Unity Catalog so that we can utilize Unity Catalog's full governance for the model.

In [0]:
import mlflow

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

## Step 1: Register the Model in Unity Catalog

Registering a model in Unity Catalog:
- Creates a versioned, governed model artifact
- Links the model back to the MLflow run (automatic lineage!)
- Enables you to assign human-readable aliases like `@champion`

In [0]:
# Register the pyfunc / feature-store artifact as @champion.
# This version carries feature-lookup metadata so fe.score_batch() can retrieve
# feature values automatically from the offline feature table by customerID.
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=model_name,
)

print(f"✓ Registered: {registered.name}")
print(f"  Version   : {registered.version}")
print(f"  Status    : {registered.status}")

In [0]:
from mlflow import MlflowClient
# Instantiate a MLflow client
client = MlflowClient()

# Assign the 'champion' alias — this is what Model Serving and batch jobs will reference
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered.version,
)
print(f"✓ Alias 'champion' set on version {registered.version}")
print(f"\nModel URI with alias: models:/{model_name}@champion")

### Sklearn pipeline
Also register the raw sklearn pipeline for the serving endpoint.

In a production real-time implementation, you would want serve the pyfunc artifact (`@champion`) on a model serving endpoint. This would need a Unity Catalog online Synched table for real-time feature lookup — an infrastructure component we don't set up in this workshop in the interest of time.  

We will instead serve the sklearn pipeline, which receives full features as input.

<img src=../utils/resources/two_alias_pattern.png>

In [0]:
# Also register the raw sklearn pipeline for the serving endpoint.
# The pyfunc artifact (@champion) needs an online Synched able for real-time feature lookup —
# an infrastructure component we don't set up in this workshop.
# The raw sklearn pipeline receives full features as input and deploys without the synched table.
serving_registered = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model_pipeline",
    name=model_name,
)

client.set_registered_model_alias(
    name=model_name,
    alias="serving",
    version=serving_registered.version,
)

print(f"✓ Alias '@serving' set on version {serving_registered.version}  (raw sklearn pipeline)")
print()
print(f"  @champion  → v{registered.version}    pyfunc / feature-store  →  fe.score_batch()")
print(f"  @serving   → v{serving_registered.version}    raw sklearn pipeline    →  serving endpoint")

### Lineage
Unity Catalog tracks and monitors all activity for tables, views, volumes, functions, and ML models to produce full lineage and tracability between assets.

**In the UI:**
1. Open **Catalog** and navigate to **{catalog}.{your_schema}**.
1. Select the **Modles** tab and then select your **churn_classifier** model.  This will open the model and all its tracked versions in Unity Catalog.
1. Select **Version 1**.  Note all the parameters and metrics that are maintained with the model version, including the alias you assigned.
2. Click the **Lineage** tab.  This lists all the notebooks, jobs, pipelines, model serving endpoint, etc. both upstream and downstream of the model version.
Click **See lineage graph**
Explore the lineage information tracked by Unity Catalog.  Expand the upstream assets.  

- What is the earliest origin of the data?
- Is lineage tracked at the column level?

<img src=../utils/resources/lineagetree.png>

## Step 2: Create a Model Serving Endpoint

> **Batch scoring via `fe.score_batch()` (Step 3) is the primary production pattern for a churn model** — predictions are written to a Delta table and consumed by CRM/marketing systems on a nightly or weekly schedule.
>
> A **serving endpoint** is useful when you need an on-demand score: for example, a customer-support tool that needs a risk score the moment a customer calls. We show the full setup here so you can see how it's done.

<img src=../utils/resources/batch_scoring_architecture.png>

Databricks Model Serving creates a managed REST endpoint that:
- Auto-scales from zero (no cost when idle)
- Handles versioning and traffic routing
- Logs every request/response to an **AI Gateway inference table**
- **Rate-limits requests** via AI Gateway (60 calls/min per user)

We use the `@serving` version (raw sklearn pipeline) — it accepts all feature values as input and works without an Online Table.

We start creation **asynchronously** and continue immediately to Step 3. The endpoint will be ready in 5-10 minutes — you can run the REST test cell at the end of this notebook once it shows **Ready** in the Serving UI.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    EndpointCoreConfigInput,
    ServedModelInput,
)

w = WorkspaceClient()
endpoint_name = f"churn_{safe_username}"

# create() fires asynchronously — returns immediately while the endpoint spins up.
# Use create_and_wait() if you want to block until Ready (adds 5-10 minutes).
w.serving_endpoints.create(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        name=endpoint_name,
        served_models=[
            ServedModelInput(
                model_name=model_name,
                model_version=serving_registered.version,
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix="churn",
            enabled=True,
        ),
        rate_limits=[
            AiGatewayRateLimit(
                calls=60,
                key=AiGatewayRateLimitKey.USER,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            )
        ],
    ),
)
print(f"✓ Endpoint creation started: {endpoint_name}")
print(f"  Model version : {serving_registered.version}  (@serving — raw sklearn pipeline)")
print(f"  Inference table will appear at: {catalog}.{schema}.churn_payload")
print()
print("  → Continue to Step 3. Check Serving > {endpoint_name} for status.")

### Monitor Endpoint Creation (Background)

1. Click **Serving** in the left sidebar
2. Find <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_{safe_username}</span> — status will show **Creating**
3. **Continue with Step 3 below** — batch inference and the LLM explanation do not require the endpoint to be ready
4. Come back to the REST test cell (near the bottom) once status shows **Ready**

Observe the endpoint configuration when it's ready:
- Served model: `@serving` alias (raw sklearn pipeline)
- Scale-to-zero enabled
- AI Gateway: inference table + rate limits configured

## Step 3: Test


### Test the basic sklearn pipeline version 
This model takes full feature records as input and has been registered with the `@serving` alias.

In [0]:
import mlflow
import time
import numpy as np
import pandas as pd

# Load the sklearn model
start = time.time()
model = mlflow.pyfunc.load_model(f"models:/{model_name}@serving")
print(f"Model loaded in {time.time() - start:.1f}s")

sample_input = pd.DataFrame([{
    "customerID":       "TEST_001",
    "gender":           "Female",
    "SeniorCitizen":    "0",
    "Partner":          "No",
    "Dependents":       "No",
    "tenure":           np.int32(1),
    "PhoneService":     "Yes",
    "MultipleLines":    "No",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "No",
    "OnlineBackup":     "No",
    "DeviceProtection": "No",
    "TechSupport":      "No",
    "StreamingTV":      "Yes",
    "StreamingMovies":  "Yes",
    "Contract":         "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Electronic check",
    "MonthlyCharges":   85.7,
    "TotalCharges":     85.7,
},
{
    "customerID":       "TEST_002",
    "gender":           "Male",
    "SeniorCitizen":    "0",
    "Partner":          "Yes",
    "Dependents":       "Yes",
    "tenure":           np.int32(72),
    "PhoneService":     "Yes",
    "MultipleLines":    "Yes",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "Yes",
    "OnlineBackup":     "Yes",
    "DeviceProtection": "Yes",
    "TechSupport":      "Yes",
    "StreamingTV":      "Yes",
    "StreamingMovies":  "Yes",
    "Contract":         "Two year",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Credit card (automatic)",
    "MonthlyCharges":   114.05,
    "TotalCharges":     8468.2,
}])

local_predictions = model.predict(sample_input)

display(pd.DataFrame({
    "customerID": sample_input["customerID"],
    "prediction": local_predictions,
}))

#### Feature Store Batch Inference — The Primary Production Pattern

For a churn model, **batch scoring is the right architecture**. A nightly or weekly job:
1. Pulls customer IDs from your data warehouse
2. Looks up current feature values from the Feature Store (offline Delta table)
3. Scores all customers in a single Spark job
4. Writes predictions to a Delta table
5. CRM / marketing systems read predictions from that table

`fe.score_batch()` implements this in one call — it handles the feature lookup automatically.

The two UC aliases map directly to the two deployment patterns:

| Alias | Artifact | Pattern |
|---|---|---|
| `@champion` | `model/` (pyfunc + feature-store) | `fe.score_batch()` — key-based lookup from offline Delta table |
| `@serving` | `model_pipeline/` (raw sklearn) | Serving endpoint — receives full features via REST |

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Only the lookup key is needed — feature values are fetched from the feature store automatically
feature_table = f"{catalog}.{schema}.churn_features"
customer_ids_df = spark.table(feature_table).limit(5).select("customerID")

display(customer_ids_df)

predictions_df = fe.score_batch(
    model_uri=f"models:/{model_name}@champion",
    df=customer_ids_df,
)

display(predictions_df.select("customerID", "prediction"))

#### REST Interface

> **Run this once the endpoint is ready.** Check **Serving** > `churn_{safe_username}` — status must show **Ready**. If it's still **Creating**, skip to the next module — it uses a local prediction from the pyfunc model and therefore doesn't need the endpoint.

This shows how any application can call the model over HTTP using the `@serving` version (raw sklearn pipeline). The request body includes all feature values; the endpoint scores and returns predictions immediately.


In [0]:
import requests, json

# Get workspace URL and auth headers (works with OAuth, PAT, or any auth method)
w_client = WorkspaceClient()
workspace_url = w_client.config.host
headers = {"Content-Type": "application/json"}
headers.update(w_client.config.authenticate())

# Sample customer: new month-to-month customer, high monthly charges — high churn risk
sample_customer = {
    "dataframe_records": [{
        "customerID":       "NEW_CUSTOMER_001",
        "gender":           "Female",
        "SeniorCitizen":    0,
        "Partner":          "No",
        "Dependents":       "No",
        "tenure":           1,
        "PhoneService":     "Yes",
        "MultipleLines":    "No",
        "InternetService":  "Fiber optic",
        "OnlineSecurity":   "No",
        "OnlineBackup":     "No",
        "DeviceProtection": "No",
        "TechSupport":      "No",
        "StreamingTV":      "Yes",
        "StreamingMovies":  "Yes",
        "Contract":         "Month-to-month",
        "PaperlessBilling": "Yes",
        "PaymentMethod":    "Electronic check",
        "MonthlyCharges":   85.7,
        "TotalCharges":     85.7,
    }]
}

response = requests.post(
    f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations",
    headers=headers,
    json=sample_customer,
    timeout=60,
)

result = response.json()
print(f"Status: {response.status_code}")
print(f"Prediction: {json.dumps(result, indent=2)}")

## Summary

In this notebook you:

1. Registered the best model in Unity Catalog with `@champion` and `@serving` aliases
2. Deployed a rate-limited serving endpoint with AI Gateway
3. Scored customers 
   1. locally via sklearn, 
   2. batch via pyfunc + Feature Store, 
   3. and on-demand via REST


➡️ Next: [06_llm_explanation.ipynb]($./06_llm_explanation)